In [3]:
# %% [markdown]
# # Inforcer API Ingestion to Fabric Lakehouse
# ## Complete Solution with Incremental Load & Pagination Support

# %% [code]
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, current_timestamp, lit, explode, posexplode
from pyspark.sql.types import StructType, MapType, ArrayType, StringType, TimestampType, LongType
import requests
from datetime import datetime, timezone
from typing import Dict, List, Optional
import time
import json
from delta.tables import DeltaTable

# ============================================
# CONFIGURATION
# ============================================

class InforcerConfig:
    # Region is set to US only
    REGION = 'us'
    
    BASE_URLS = {
        'us': 'https://api-us.inforcer.com/api'
    }
    
    BASE_URL = BASE_URLS[REGION]
    API_KEY = "0ce9a0ed6d1e4df998e62e57633a4f17"  # Replace with your actual API key
    
    # Lakehouse configuration
    LAKEHOUSE_NAME = "ManagedServiceData"  # Replace with your lakehouse name
    
    # Incremental load configuration
    CHECKPOINT_TABLE = "ingestion_checkpoints"
    
    # Rate limiting (keep below Inforcer's 100 calls/minute limit)
    REQUESTS_PER_MINUTE = 90
    REQUEST_DELAY = 60 / REQUESTS_PER_MINUTE

config = InforcerConfig()
spark = SparkSession.builder.getOrCreate()

# ============================================
# API CLIENT WITH PAGINATION
# ============================================

class InforcerAPIClient:
    """Handles API calls with pagination and rate limiting"""
    
    def __init__(self, config):
        self.base_url = config.BASE_URL
        self.api_key = config.API_KEY
        self.request_delay = config.REQUEST_DELAY
        self.last_request_time = 0
    
    def _make_request(self, endpoint: str, params: Dict = None) -> Dict:
        """Make API request with rate limiting and retry logic"""
        # Rate limiting
        current_time = time.time()
        time_since_last = current_time - self.last_request_time
        if time_since_last < self.request_delay:
            time.sleep(self.request_delay - time_since_last)
        
        url = f"{self.base_url}{endpoint}"
        headers = {"Inf-Api-Key": self.api_key}
        
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = requests.get(url, headers=headers, params=params)
                self.last_request_time = time.time()
                
                if response.status_code == 200:
                    json_data = response.json()
                    return json_data
                elif response.status_code == 429:
                    # Rate limit exceeded - wait and retry
                    wait_time = 60 * (attempt + 1)
                    print(f"Rate limit hit. Waiting {wait_time} seconds...")
                    time.sleep(wait_time)
                    continue
                else:
                    error_msg = f"API Error {response.status_code}: {response.text}"
                    print(error_msg)
                    return {"error": error_msg, "status": "error"}
            except Exception as e:
                print(f"Request attempt {attempt + 1} failed: {str(e)}")
                if attempt == max_retries - 1:
                    return {"error": str(e), "status": "error"}
                time.sleep(5)
        
        return {"error": "Max retries exceeded", "status": "error"}
    
    def get_with_pagination(self, endpoint: str, page_param: str = "page", 
                           size_param: str = "pageSize", page_size: int = 100) -> List[Dict]:
        """Handle pagination for endpoints that support it"""
        all_data = []
        page = 1
        
        while True:
            params = {page_param: page, size_param: page_size}
            response = self._make_request(endpoint, params)
            
            if response.get("status") == "error":
                break
            
            data = response.get("data", [])
            if not data:
                break
                
            all_data.extend(data)
            
            # Check if we've received all pages
            if len(data) < page_size:
                break
                
            page += 1
            print(f"  Fetched page {page-1}, total records: {len(all_data)}")
        
        return all_data
    
    def get_single(self, endpoint: str) -> Optional[Dict]:
        """Get single endpoint data"""
        response = self._make_request(endpoint)
        if response.get("status") == "success":
            return response.get("data")
        elif response.get("success") == True:
            # Handle {"success": true, "data": {...}} format
            return response.get("data")
        elif isinstance(response, dict) and "error" not in response and "data" not in response:
            # If no wrapper, return the response directly
            return response
        return None
    
    def get_list(self, endpoint: str) -> List[Dict]:
        """Get list data (non-paginated)"""
        response = self._make_request(endpoint)
        
        if isinstance(response, list):
            return response
        elif isinstance(response, dict):
            if response.get("success") == True or response.get("status") == "success":
                data = response.get("data", [])
                return data if isinstance(data, list) else []
            elif response.get("status") == "error":
                return []
            elif "data" in response:
                data = response.get("data", [])
                return data if isinstance(data, list) else []
            else:
                return [response] if response else []
        
        return []
    
    def get_all_tenants(self) -> List[Dict]:
        """Fetch every tenant from /beta/tenants (licensed + assessment customers).

        The endpoint currently returns the full tenant set in a single response,
        but we page defensively so the ingestion keeps working if Inforcer adds
        pagination later. We de-duplicate on clientTenantId and stop as soon as a
        page returns no new tenants (the API currently ignores page params and
        repeats the same set), which prevents an infinite loop.
        """
        all_tenants: List[Dict] = []
        seen_ids = set()
        page = 1
        page_size = 100
        while True:
            resp = self._make_request("/beta/tenants", {"page": page, "pageSize": page_size})
            data = resp.get("data", []) if isinstance(resp, dict) else (resp or [])
            if not data:
                break
            new_records = [t for t in data if t.get("clientTenantId") not in seen_ids]
            for t in new_records:
                seen_ids.add(t.get("clientTenantId"))
            all_tenants.extend(new_records)
            # Stop when the page adds nothing new (API repeats the set) or is a short/last page
            if not new_records or len(data) < page_size:
                break
            page += 1
        return all_tenants
    
    def search_audit_events(self, search_criteria: Dict) -> List[Dict]:
        """Search audit events with criteria"""
        endpoint = "/beta/auditEvents/search"
        response = self._make_request(endpoint, params=search_criteria)
        
        if isinstance(response, list):
            return response
        elif isinstance(response, dict):
            if response.get("success") == True or response.get("status") == "success":
                data = response.get("data", [])
                return data if isinstance(data, list) else []
            elif "data" in response:
                data = response.get("data", [])
                return data if isinstance(data, list) else []
        
        return []

# ============================================
# HELPER FUNCTIONS
# ============================================

def flatten_dict(d: Dict, parent_key: str = '', sep: str = '_') -> Dict:
    """
    Recursively flatten a nested dictionary
    Example: {'a': {'b': 1}} -> {'a_b': 1}
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def normalize_data_for_spark(data: List[Dict], extract_arrays: bool = False) -> tuple:
    """
    Flatten all nested structures for better data analysis
    - Nested dicts are flattened with underscore-separated field names
    - Arrays of objects are extracted to separate tables when extract_arrays=True
    - Arrays of primitives are converted to pipe-delimited strings
    Returns: (main_data, extracted_arrays_dict)
    """
    if not data:
        return [], {}
    
    # Identify columns that have all None values
    all_keys = set()
    for record in data:
        all_keys.update(record.keys())
    
    columns_with_values = set()
    for key in all_keys:
        for record in data:
            if record.get(key) is not None:
                columns_with_values.add(key)
                break
    
    # Identify array columns for extraction
    array_columns = {}
    if extract_arrays:
        for key in columns_with_values:
            for record in data:
                value = record.get(key)
                # Only extract arrays of objects (dicts), not primitive arrays
                if value is not None and isinstance(value, list) and len(value) > 0:
                    if isinstance(value[0], dict):
                        array_columns[key] = True
                    break
    
    # Normalize main data
    normalized = []
    extracted_arrays = {}
    
    for record in data:
        normalized_record = {}
        
        for key, value in record.items():
            # Skip columns that are all None
            if key not in columns_with_values:
                continue
            
            # Handle arrays of objects - extract to separate structure if requested
            if extract_arrays and key in array_columns and isinstance(value, list):
                # Store array items for separate table
                if key not in extracted_arrays:
                    extracted_arrays[key] = []
                
                # Get parent ID for relationship
                parent_id = record.get('clientTenantId') or record.get('id')
                
                for item in value:
                    extracted_item = item.copy() if isinstance(item, dict) else {'value': item}
                    extracted_item['parent_id'] = parent_id
                    extracted_arrays[key].append(extracted_item)
                
                # Don't include array in main record
                continue
            elif isinstance(value, dict):
                # Flatten nested dictionaries
                flattened = flatten_dict({key: value})
                normalized_record.update(flattened)
            elif isinstance(value, list):
                # Convert arrays of primitives to pipe-delimited string
                if len(value) == 0:
                    normalized_record[key] = None
                elif all(isinstance(item, (str, int, float, bool, type(None))) for item in value):
                    # Array of primitives - convert to delimited string
                    normalized_record[key] = '|'.join(str(item) for item in value if item is not None)
                else:
                    # Complex array that wasn't extracted - convert to JSON as fallback
                    normalized_record[key] = json.dumps(value)
            elif value == "":
                # Replace empty strings with None for better schema inference
                normalized_record[key] = None
            else:
                normalized_record[key] = value
        
        # Only add record if it has at least one non-null value
        if any(v is not None for v in normalized_record.values()):
            normalized.append(normalized_record)
    
    return normalized, extracted_arrays

# ============================================
# LAKEHOUSE MANAGER WITH INCREMENTAL LOAD
# ============================================

class LakehouseManager:
    """Manages Lakehouse operations and incremental loading"""
    
    def __init__(self, lakehouse_name: str):
        self.lakehouse_name = lakehouse_name
        self.checkpoint_table = config.CHECKPOINT_TABLE
        self._init_checkpoint_table()
    
    def _init_checkpoint_table(self):
        """Initialize checkpoint table for incremental loads"""
        if not spark.catalog.tableExists(self.checkpoint_table):
            from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
            
            checkpoint_schema = StructType([
                StructField("endpoint", StringType(), True),
                StructField("last_load_timestamp", TimestampType(), True),
                StructField("last_load_date", StringType(), True),
                StructField("last_record_count", LongType(), True)
            ])
            
            checkpoint_df = spark.createDataFrame([], checkpoint_schema)
            checkpoint_df.write.mode("overwrite").format("delta").saveAsTable(self.checkpoint_table)
            print(f"✅ Created checkpoint table: {self.checkpoint_table}")
    
    def get_last_load_time(self, endpoint: str) -> Optional[datetime]:
        """Get last successful load timestamp for an endpoint"""
        checkpoint_df = spark.table(self.checkpoint_table)
        last_load = checkpoint_df.filter(col("endpoint") == endpoint).select("last_load_timestamp").collect()
        
        if last_load and last_load[0][0]:
            return last_load[0][0]
        return None
    
    def update_checkpoint(self, endpoint: str, load_time: datetime, record_count: int):
        """Update checkpoint after successful load"""
        if not spark.catalog.tableExists(self.checkpoint_table):
            self._init_checkpoint_table()
        
        delta_table = DeltaTable.forName(spark, self.checkpoint_table)
        
        new_checkpoint = spark.createDataFrame(
            [(endpoint, load_time, str(load_time.date()), int(record_count))],
            ["endpoint", "last_load_timestamp", "last_load_date", "last_record_count"]
        )
        
        delta_table.alias("target").merge(
            new_checkpoint.alias("source"),
            "target.endpoint = source.endpoint"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    
    def write_incremental(self, df: DataFrame, table_name: str, 
                         key_columns: List[str], endpoint: str) -> int:
        """Write data incrementally using merge/upsert"""
        if df.isEmpty():
            print(f"⚠️ No data to write for {table_name}")
            return 0
        
        df = df.withColumn("_ingested_at", current_timestamp())
        df = df.withColumn("_source_endpoint", lit(endpoint))
        
        record_count = df.count()
        
        if spark.catalog.tableExists(table_name):
            # For schema evolution, use overwrite mode if it's a major schema change
            # Check if this is the tenants table with extracted arrays
            if table_name == "inforcer_tenants" and endpoint == "/beta/tenants":
                # Drop and recreate for clean schema
                spark.sql(f"DROP TABLE IF EXISTS {table_name}")
                df.write.mode("overwrite").format("delta").saveAsTable(table_name)
                print(f"✅ {table_name}: Recreated table with {record_count} records (schema updated)")
            elif table_name == "inforcer_tenant_details":
                # Drop and recreate for tenant details (API response format fix)
                spark.sql(f"DROP TABLE IF EXISTS {table_name}")
                df.write.mode("overwrite").format("delta").saveAsTable(table_name)
                print(f"✅ {table_name}: Recreated table with {record_count} records (schema updated)")
            elif table_name == "inforcer_assessments":
                # Drop and recreate for assessments (enriched with tenant data)
                spark.sql(f"DROP TABLE IF EXISTS {table_name}")
                df.write.mode("overwrite").format("delta").saveAsTable(table_name)
                print(f"✅ {table_name}: Recreated table with {record_count} records (schema updated)")
            else:
                # Use merge with schema evolution for other tables
                delta_table = DeltaTable.forName(spark, table_name)
                merge_condition = " AND ".join([f"target.{col} = source.{col}" for col in key_columns])
                
        """Ingest all tenants with separate tables for tags and alignmentSummaries.

        Pulls every tenant exposed by /beta/tenants (licensed + assessment
        customers). Each tenant row keeps its DNS name (tenantDnsName) and gains a
        flattened license summary + a tenant_type flag so the licensed vs.
        assessment distinction is queryable directly on the main table.
        """
        print("📋 Ingesting tenants (licensed + assessment customers)...")
        tenants = self.client.get_all_tenants()
        
        if not tenants or len(tenants) == 0:
            print("  ⚠️ WARNING: No tenants returned from API!")
            return 0
        
        print(f"  ✅ Retrieved {len(tenants)} tenants from /beta/tenants")
        
        # Keep licensing visible on the main tenant row (the full per-license
        # breakdown is still extracted to inforcer_tenant_licenses below).
        for t in tenants:
            licenses = t.get("licenses") or []
            skus = [l.get("sku") for l in licenses if isinstance(l, dict) and l.get("sku")]
            t["licenseSkus"] = "|".join(skus) if skus else None
            t["tenant_type"] = "Licensed" if skus else "Assessment"
        
        # Normalize with array extraction
        normalized_tenants, extracted_arrays = normalize_data_for_spark(tenants, extract_arrays=True)
        
        if not normalized_tenants:
            print("  ⚠️ WARNING: No valid tenant data after normalization!")
            return 0
        
        # Ingest main tenant data
        df = spark.createDataFrame(normalized_tenants)
        count = self.lakehouse.write_incremental(
            df, "inforcer_tenants", ["clientTenantId"], "/beta/tenants"
        )
        # Normalize with array extraction
        # Ingest extracted arrays as separate tables
        if 'tags' in extracted_arrays and extracted_arrays['tags']:
            tags_df = spark.createDataFrame(extracted_arrays['tags'])
            self.lakehouse.write_incremental(
                tags_df, "inforcer_tenant_tags", ["parent_id", "id"], "/beta/tenants/tags"
            )
            print(f"  ✅ Extracted {len(extracted_arrays['tags'])} tags to separate table")
        
        if 'alignmentSummaries' in extracted_arrays and extracted_arrays['alignmentSummaries']:
            alignment_df = spark.createDataFrame(extracted_arrays['alignmentSummaries'])
            self.lakehouse.write_incremental(
                alignment_df, "inforcer_tenant_alignment_summaries", 
                ["parent_id", "alignedBaselineId"], "/beta/tenants/alignmentSummaries"
            )
            print(f"  ✅ Extracted {len(extracted_arrays['alignmentSummaries'])} alignment summaries to separate table")
        
        if 'licenses' in extracted_arrays and extracted_arrays['licenses']:
            licenses_df = spark.createDataFrame(extracted_arrays['licenses'])
            self.lakehouse.write_incremental(
                licenses_df, "inforcer_tenant_licenses", ["parent_id", "sku"], "/beta/tenants/licenses"
            )
            print(f"  ✅ Extracted {len(extracted_arrays['licenses'])} licenses to separate table")
        
        return count
    
    def ingest_policies(self, tenant_id: str):
        """Ingest policies for a specific tenant"""
        print(f"  📄 Ingesting policies for tenant {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/policies"
        policies = self.client.get_list(endpoint)
        
        if not policies or len(policies) == 0:
            return 0
        
        for policy in policies:
            policy['tenant_id'] = tenant_id
    
        try:
            normalized_policies, _ = normalize_data_for_spark(policies)
            if normalized_policies:
                df = spark.createDataFrame(normalized_policies)
                return self.lakehouse.write_incremental(
                    df, "inforcer_policies", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting policies: {str(e)}")
        return 0
    
    def ingest_assessments(self):
        """Ingest assessments with tenant details (name, ID) and initiating user"""
        print("📋 Ingesting assessments...")
        assessments = self.client.get_list("/beta/assessments")
        
        if not assessments or len(assessments) == 0:
            print("  ⚠️ No valid assessments data returned")
            return 0
        
        try:
            # Normalize assessment data first
            normalized_assessments, _ = normalize_data_for_spark(assessments)
            if not normalized_assessments:
                return 0
            
            # Create DataFrame
            df = spark.createDataFrame(normalized_assessments)
            
            # Enrich with tenant information if tenants table exists
            if spark.catalog.tableExists("inforcer_tenants"):
                tenants_df = spark.table("inforcer_tenants").select(
                    col("clientTenantId").alias("tenantId_lookup"),
                    col("tenantFriendlyName").alias("assessed_tenant_name"),
                    col("msTenantId").alias("assessed_tenant_ms_id")
                )
                
                # Determine which column in assessments contains the tenant reference
                # Check for common tenant ID column names
                tenant_ref_col = None
                for possible_col in ["tenantId", "clientTenantId", "tenant_id", "msTenantId"]:
                    if possible_col in df.columns:
                        tenant_ref_col = possible_col
                        break
                
                if tenant_ref_col:
                    # Join assessments with tenant data
                    df = df.join(
                        tenants_df,
                        df[tenant_ref_col] == tenants_df["tenantId_lookup"],
                        "left"
                    ).drop("tenantId_lookup")
                    print(f"  ✅ Enriched with tenant details using '{tenant_ref_col}' field")
                else:
                    print("  ⚠️ No tenant reference field found in assessments")
            
            # Add user information column (if exists in the data)
            # Common field names: createdBy, initiatedBy, userId, createdByUserId, etc.
            user_field_found = False
            for user_col in ["createdBy", "initiatedBy", "userId", "createdByUserId", "requestedBy"]:
                if user_col in df.columns:
                    df = df.withColumnRenamed(user_col, "assessment_initiated_by")
                    user_field_found = True
                    print(f"  ✅ Using '{user_col}' as assessment initiator")
                    break
            
            if not user_field_found:
                # Add placeholder column if not found
                df = df.withColumn("assessment_initiated_by", lit(None).cast(StringType()))
                print("  ⚠️ No user initiator field found, added null column")
            
            return self.lakehouse.write_incremental(
                df, "inforcer_assessments", ["id"], "/beta/assessments"
            )
        except Exception as e:
            print(f"  ⚠️ Error creating DataFrame: {str(e)}")
        return 0
    
    def ingest_alignment_scores(self):
        """Ingest alignment scores"""
        print("📊 Ingesting alignment scores...")
        scores = self.client.get_list("/beta/alignmentScores")
        
        if not scores or len(scores) == 0:
            print("  ⚠️ No valid alignment scores data returned")
            return 0
        except Exception as e:
        try:
            normalized_scores, _ = normalize_data_for_spark(scores)
            if normalized_scores:
                df = spark.createDataFrame(normalized_scores)
                return self.lakehouse.write_incremental(
                    df, "inforcer_alignment_scores", ["id"], "/beta/alignmentScores"
                )
        except Exception as e:
            print(f"  ⚠️ Error creating DataFrame: {str(e)}")
        return 0
    
    def ingest_baselines(self):
        """Ingest baseline information"""
        print("📏 Ingesting baselines...")
        baselines = self.client.get_list("/beta/baselines")
        
        if not baselines or len(baselines) == 0:
            print("  ⚠️ No valid baseline data returned")
            return 0
        except Exception as e:
        try:
            normalized_baselines, _ = normalize_data_for_spark(baselines)
            if normalized_baselines:
                df = spark.createDataFrame(normalized_baselines)
                return self.lakehouse.write_incremental(
                    df, "inforcer_baselines", ["id"], "/beta/baselines"
                )
        except Exception as e:
            print(f"  ⚠️ Error creating DataFrame: {str(e)}")
        return 0
    
    def ingest_audit_event_types(self):
        """Ingest audit event types"""
        print("📝 Ingesting audit event types...")
        event_types = self.client.get_list("/beta/auditEvents/eventTypes")
        
        if not event_types or len(event_types) == 0:
            print("  ⚠️ No valid audit event types data returned")
            return 0
        
        try:
            event_data = [{"event_type": et} for et in event_types if et]
            if event_data:
                df = spark.createDataFrame(event_data)
                return self.lakehouse.write_incremental(
                    df, "inforcer_audit_event_types", ["event_type"], "/beta/auditEvents/eventTypes"
                )
        except Exception as e:
            print(f"  ⚠️ Error creating DataFrame: {str(e)}")
        return 0
    
    def ingest_audit_events_search(self, search_criteria: Dict = None):
        """Search and ingest audit events based on criteria"""
        print("🔍 Searching audit events...")
        
        if search_criteria is None:
            from datetime import timedelta
            end_date = datetime.now(timezone.utc)
            start_date = end_date - timedelta(days=30)
            search_criteria = {
                "startDate": start_date.isoformat(),
                "endDate": end_date.isoformat()
            }
        
        events = self.client.search_audit_events(search_criteria)
        
        if not events or len(events) == 0:
            print("  ⚠️ No audit events returned")
            return 0
        
        try:
            normalized_events, _ = normalize_data_for_spark(events)
            if normalized_events:
                df = spark.createDataFrame(normalized_events)
                return self.lakehouse.write_incremental(
                    df, "inforcer_audit_events", ["id"], "/beta/auditEvents/search"
                )
        except Exception as e:
            print(f"  ⚠️ Error creating DataFrame: {str(e)}")
        return 0
    
    def ingest_tenant_users(self, tenant_id: str):
        """Ingest users for a specific tenant"""
        print(f"  👤 Ingesting users for tenant {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/users"
        users = self.client.get_list(endpoint)
        
        if not users or len(users) == 0:
            return 0
        
        for user in users:
            user['tenant_id'] = tenant_id
        
        try:
            normalized_users, _ = normalize_data_for_spark(users)
            if normalized_users:
                df = spark.createDataFrame(normalized_users)
                return self.lakehouse.write_incremental(
                    df, "inforcer_users", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting users: {str(e)}")
        return 0
    
    def ingest_tenant_user_details(self, tenant_id: str, user_id: str):
        """Ingest details for a specific user in a tenant"""
        print(f"  👤 Ingesting user details for tenant {tenant_id}, user {user_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/users/{user_id}"
        user_details = self.client.get_single(endpoint)
        
        if not user_details:
            return 0
        
        user_details['tenant_id'] = tenant_id
        
        try:
            normalized_details, _ = normalize_data_for_spark([user_details])
            if normalized_details:
                df = spark.createDataFrame(normalized_details)
                return self.lakehouse.write_incremental(
                    df, "inforcer_user_details", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting user details: {str(e)}")
        return 0
    
    def ingest_tenant_groups(self, tenant_id: str):
        """Ingest groups for a specific tenant"""
        print(f"  👥 Ingesting groups for tenant {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/groups"
        groups = self.client.get_with_pagination(endpoint, page_size=100)
        
        if not groups or len(groups) == 0:
            return 0
        
        for group in groups:
            group['tenant_id'] = tenant_id
        
        try:
            normalized_groups, _ = normalize_data_for_spark(groups)
            if normalized_groups:
                df = spark.createDataFrame(normalized_groups)
                return self.lakehouse.write_incremental(
                    df, "inforcer_groups", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting groups: {str(e)}")
        return 0
    
    def ingest_tenant_group_details(self, tenant_id: str, group_id: str):
        """Ingest details for a specific group in a tenant"""
        print(f"  👥 Ingesting group details for tenant {tenant_id}, group {group_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/groups/{group_id}"
        group_details = self.client.get_single(endpoint)
        
        if not group_details:
            return 0
        
        group_details['tenant_id'] = tenant_id
        
        try:
            normalized_details, _ = normalize_data_for_spark([group_details])
            if normalized_details:
                df = spark.createDataFrame(normalized_details)
                return self.lakehouse.write_incremental(
                    df, "inforcer_group_details", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting group details: {str(e)}")
        return 0
    
    def ingest_tenant_roles(self, tenant_id: str):
        """Ingest roles for a specific tenant"""
        print(f"  🎭 Ingesting roles for tenant {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/roles"
        roles = self.client.get_list(endpoint)
        
        if not roles or len(roles) == 0:
            return 0
        
        for role in roles:
            role['tenant_id'] = tenant_id
        
        try:
            normalized_roles, _ = normalize_data_for_spark(roles)
            if normalized_roles:
                df = spark.createDataFrame(normalized_roles)
                return self.lakehouse.write_incremental(
                    df, "inforcer_roles", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting roles: {str(e)}")
        return 0
    
    def ingest_tenant_alignment_details(self, tenant_id: str):
        """Ingest alignment details for a specific tenant"""
        print(f"  🎯 Ingesting alignment details for tenant {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/alignmentDetails"
        details = self.client.get_single(endpoint)
        
        if not details:
            return 0
        
        details['tenant_id'] = tenant_id
        
        try:
            normalized_details, _ = normalize_data_for_spark([details])
            if normalized_details:
                df = spark.createDataFrame(normalized_details)
                return self.lakehouse.write_incremental(
                    df, "inforcer_alignment_details", ["tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting alignment details: {str(e)}")
        return 0
    
    def ingest_tenant_details(self, tenant_id: str):
        """Ingest full details for a specific tenant"""
        print(f"  📄 Ingesting tenant details for {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}"
        details = self.client.get_single(endpoint)
        
        if not details:
            return 0
        
        try:
            normalized_details, _ = normalize_data_for_spark([details])
            if normalized_details:
                df = spark.createDataFrame(normalized_details)
                return self.lakehouse.write_incremental(
                    df, "inforcer_tenant_details", ["clientTenantId"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting tenant details: {str(e)}")
        return 0
    
    def ingest_tenant_secure_scores(self, tenant_id: str):
        """Ingest secure scores for a specific tenant"""
        print(f"  🔒 Ingesting secure scores for tenant {tenant_id}...")
        endpoint = f"/beta/tenants/{tenant_id}/secureScores"
        scores = self.client.get_list(endpoint)
        
        if not scores or len(scores) == 0:
            return 0
        
        for score in scores:
            score['tenant_id'] = tenant_id
        
        try:
            normalized_scores, _ = normalize_data_for_spark(scores)
            if normalized_scores:
                df = spark.createDataFrame(normalized_scores)
                return self.lakehouse.write_incremental(
                    df, "inforcer_tenant_secure_scores", ["id", "tenant_id"], endpoint
                )
        except Exception as e:
            print(f"  ⚠️ Error ingesting secure scores: {str(e)}")
        return 0

# ============================================
# MAIN EXECUTION
# ============================================

def run_full_ingestion():
    """Run complete data ingestion for all endpoints"""
    print("="*60)
    print("🚀 Starting Inforcer API Ingestion Pipeline")
    print("="*60)
    print(f"📍 Base URL: {config.BASE_URL}")
    print(f"🏗️  Lakehouse: {config.LAKEHOUSE_NAME}")
    print(f"⏱️  Started at: {datetime.now()}")
    print("="*60)
    
    client = InforcerAPIClient(config)
    lakehouse = LakehouseManager(config.LAKEHOUSE_NAME)
    ingestion = InforcerIngestion(client, lakehouse)
    
    # Step 1: Ingest tenants first
    print("\n📌 STEP 1: Ingesting Tenants")
    print("-"*40)
    tenants_count = ingestion.ingest_tenants()
    print(f"✓ Ingested {tenants_count} tenants")
    
    # Get list of tenant IDs
    if spark.catalog.tableExists("inforcer_tenants"):
        tenants_df = spark.table("inforcer_tenants")
        tenant_ids = [str(row.clientTenantId) for row in tenants_df.select("clientTenantId").collect()]
        print(f"✓ Found {len(tenant_ids)} tenants to process")
    else:
        print("⚠️ No tenants found. Skipping tenant-specific endpoints.")
        tenant_ids = []
    
    # Step 2: Ingest global data (assessments enriched with tenant data from Step 1)
    print("\n📌 STEP 2: Ingesting Global Data")
    print("-"*40)
    ingestion.ingest_assessments()  # Enriched with tenant details
    ingestion.ingest_alignment_scores()
    ingestion.ingest_baselines()
    ingestion.ingest_audit_event_types()
    ingestion.ingest_audit_events_search()
    
    # Step 3: Ingest tenant-specific data
    if tenant_ids:
        print(f"\n📌 STEP 3: Ingesting Tenant-Specific Data for {len(tenant_ids)} Tenants")
        print("-"*40)
        
        for idx, tenant_id in enumerate(tenant_ids, 1):
            print(f"\n▶️  Processing tenant {idx}/{len(tenant_ids)}: {tenant_id}")
            ingestion.ingest_tenant_details(tenant_id)
            ingestion.ingest_policies(tenant_id)
            ingestion.ingest_tenant_users(tenant_id)
            ingestion.ingest_tenant_groups(tenant_id)
            ingestion.ingest_tenant_roles(tenant_id)
            ingestion.ingest_tenant_alignment_details(tenant_id)
            ingestion.ingest_tenant_secure_scores(tenant_id)
    else:
        print("\n⚠️ SKIPPED: No tenant-specific data to ingest")
    
    # Step 4: Summary
    print("\n" + "="*60)
    print("📊 INGESTION SUMMARY")
    print("="*60)
    
    summary_tables = [
        "inforcer_tenants",
        "inforcer_tenant_tags",
        "inforcer_tenant_alignment_summaries",
        "inforcer_tenant_licenses",
        "inforcer_tenant_details",
        "inforcer_assessments",
        "inforcer_policies", 
        "inforcer_alignment_scores",
        "inforcer_baselines",
        "inforcer_audit_event_types",
        "inforcer_audit_events",
        "inforcer_users",
        "inforcer_user_details",
        "inforcer_groups",
        "inforcer_group_details",
        "inforcer_roles",
        "inforcer_alignment_details",
        "inforcer_tenant_secure_scores"
    ]
    
    for table in summary_tables:
        if spark.catalog.tableExists(table):
            count = spark.table(table).count()
            print(f"  📋 {table}: {count:,} records")
        else:
            print(f"  ⚠️ {table}: Not created")
    
    # Show checkpoint summary
    print("\n📌 CHECKPOINT SUMMARY")
    print("-"*40)
    if spark.catalog.tableExists(config.CHECKPOINT_TABLE):
        spark.table(config.CHECKPOINT_TABLE).show(truncate=False)
    
    print("\n" + "="*60)
    print(f"✅ Ingestion completed at: {datetime.now()}")
    print("="*60)

# ============================================
# INCREMENTAL REFRESH FUNCTION
# ============================================

def incremental_refresh():
    """Perform incremental refresh based on checkpoints"""
    print("🔄 Starting incremental refresh...")
    print("="*40)
    
    if not spark.catalog.tableExists(config.CHECKPOINT_TABLE):
        print("No checkpoints found. Running full ingestion instead.")
        run_full_ingestion()
        return
    
    client = InforcerAPIClient(config)
    lakehouse = LakehouseManager(config.LAKEHOUSE_NAME)
    ingestion = InforcerIngestion(client, lakehouse)
    
    checkpoints = spark.table(config.CHECKPOINT_TABLE)
    endpoints = [row.endpoint for row in checkpoints.select("endpoint").collect() if row.endpoint]
    
    print(f"Found {len(endpoints)} endpoints with checkpoints")
    
    for endpoint in endpoints:
        print(f"\n🔄 Refreshing: {endpoint}")
        
        if endpoint == "/beta/tenants":
            ingestion.ingest_tenants()
        elif endpoint == "/beta/alignmentScores":
            ingestion.ingest_alignment_scores()
        elif endpoint == "/beta/baselines":
            ingestion.ingest_baselines()
        elif endpoint == "/beta/auditEvents/eventTypes":
            ingestion.ingest_audit_event_types()
        elif endpoint == "/beta/auditEvents/search":
            ingestion.ingest_audit_events_search()
        elif "policies" in endpoint:
            if spark.catalog.tableExists("inforcer_tenants"):
                tenants = spark.table("inforcer_tenants").select("clientTenantId").collect()
                for tenant in tenants:
                    ingestion.ingest_policies(str(tenant.clientTenantId))
        elif "users" in endpoint and "users/" not in endpoint:
            if spark.catalog.tableExists("inforcer_tenants"):
                tenants = spark.table("inforcer_tenants").select("clientTenantId").collect()
                for tenant in tenants:
                    ingestion.ingest_tenant_users(str(tenant.clientTenantId))
        elif "groups" in endpoint and "groups/" not in endpoint:
            if spark.catalog.tableExists("inforcer_tenants"):
                tenants = spark.table("inforcer_tenants").select("clientTenantId").collect()
                for tenant in tenants:
                    ingestion.ingest_tenant_groups(str(tenant.clientTenantId))
        elif "roles" in endpoint:
            if spark.catalog.tableExists("inforcer_tenants"):
                tenants = spark.table("inforcer_tenants").select("clientTenantId").collect()
                for tenant in tenants:
                    ingestion.ingest_tenant_roles(str(tenant.clientTenantId))
    
    print("\n✅ Incremental refresh complete!")

# ============================================
# VALIDATION FUNCTION
# ============================================

def validate_ingestion():
    """Validate data quality and completeness"""
    print("\n🔍 VALIDATION REPORT")
    print("="*60)
    
    tables_to_validate = [
        ("inforcer_tenants", ["clientTenantId"]),
        ("inforcer_tenant_tags", ["parent_id", "id"]),
        ("inforcer_tenant_alignment_summaries", ["parent_id"]),
        ("inforcer_tenant_details", ["clientTenantId"]),
        ("inforcer_assessments", ["id"]),
        ("inforcer_policies", ["id", "tenant_id"]),
        ("inforcer_users", ["id", "tenant_id"]),
        ("inforcer_groups", ["id", "tenant_id"]),
        ("inforcer_roles", ["id", "tenant_id"]),
        ("inforcer_tenant_secure_scores", ["id", "tenant_id"]),
    ]
    
    for table_name, key_columns in tables_to_validate:
        if spark.catalog.tableExists(table_name):
            df = spark.table(table_name)
            total_count = df.count()
            print(f"\n📋 Table: {table_name}")
            print(f"   Total records: {total_count:,}")
            
            for col_name in key_columns:
                if col_name in df.columns:
                    null_count = df.filter(col(col_name).isNull()).count()
                    if null_count > 0:
                        print(f"   ⚠️  Nulls in {col_name}: {null_count:,} ({null_count/total_count*100:.2f}%)")
                    else:
                        print(f"   ✓ No nulls in {col_name}")
            
            print(f"   Sample data:")
            df.select(key_columns).show(5, truncate=False)
        else:
            print(f"\n⚠️ Table {table_name} does not exist")
    
    print(f"\n📌 Checkpoint Table Status")
    print("-"*40)
    if spark.catalog.tableExists(config.CHECKPOINT_TABLE):
        checkpoint_df = spark.table(config.CHECKPOINT_TABLE)

        checkpoint_count = checkpoint_df.filter(col("endpoint").isNotNull()).count()
                    else:

        print(f"✓ Active checkpoints: {checkpoint_count}")
                        print(f"   ✓ No nulls in {col_name}")    validate_ingestion()

        checkpoint_df.select("endpoint", "last_load_timestamp", "last_record_count").show(truncate=False)
                run_full_ingestion()

    else:
            print(f"   Sample data:")if __name__ == "__main__":

        print("⚠️ No checkpoint table found")
            df.select(key_columns).show(5, truncate=False)

    
        else:# ============================================

    print("="*60)
            print(f"\n⚠️ Table {table_name} does not exist")# MAIN ENTRY POINT


    # ============================================

# ============================================
    print(f"\n📌 Checkpoint Table Status")

# MAIN ENTRY POINT
    print("-"*40)    print("="*60)

# ============================================
    if spark.catalog.tableExists(config.CHECKPOINT_TABLE):    


        checkpoint_df = spark.table(config.CHECKPOINT_TABLE)        print("⚠️ No checkpoint table found")

if __name__ == "__main__":
        checkpoint_count = checkpoint_df.filter(col("endpoint").isNotNull()).count()    else:

    run_full_ingestion()
        print(f"✓ Active checkpoints: {checkpoint_count}")

    validate_ingestion()        checkpoint_df.select("endpoint", "last_load_timestamp", "last_record_count").show(truncate=False)

StatementMeta(, 83a14a2f-cb13-41bf-8324-78adcbce8971, 4, Finished, Available, Finished, False)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 1037)

In [ ]:
# ============================================
# SAMPLE QUERIES - Demonstrating Flattened Data
# ============================================

print("=" * 60)
print("📊 FLATTENED DATA STRUCTURE EXAMPLES")
print("=" * 60)

# Example 1: Show flattened columns in tenants table
if spark.catalog.tableExists("inforcer_tenants"):
    print("\n1️⃣  TENANTS TABLE - Flattened Structure")
    print("-" * 60)
    tenants_df = spark.table("inforcer_tenants")
    print(f"Total columns: {len(tenants_df.columns)}")
    print(f"Column names (first 20):")
    for col in tenants_df.columns[:20]:
        print(f"  • {col}")
    print("\nSample data (first 2 records):")
    tenants_df.limit(2).show(truncate=False, vertical=True)

# Example 2: Show policies with flattened fields
if spark.catalog.tableExists("inforcer_policies"):
    print("\n2️⃣  POLICIES TABLE - Nested Dicts Flattened")
    print("-" * 60)
    policies_df = spark.table("inforcer_policies")
    print(f"Total columns: {len(policies_df.columns)}")
    # Show columns that contain underscores (indicating flattened fields)
    flattened_cols = [c for c in policies_df.columns if '_' in c and not c.startswith('_')]
    if flattened_cols:
        print(f"Flattened columns (showing nested structure):")
        for col in flattened_cols[:15]:
            print(f"  • {col}")
    policies_df.limit(1).show(truncate=False, vertical=True)

# Example 3: Show extracted arrays in separate tables
if spark.catalog.tableExists("inforcer_tenant_tags"):
    print("\n3️⃣  EXTRACTED ARRAYS - Relational Structure")
    print("-" * 60)
    tags_df = spark.table("inforcer_tenant_tags")
    print(f"Tags table: {tags_df.count()} records")
    tags_df.limit(5).show(truncate=False)

if spark.catalog.tableExists("inforcer_tenant_alignment_summaries"):
    alignment_df = spark.table("inforcer_tenant_alignment_summaries")
    print(f"\nAlignment Summaries table: {alignment_df.count()} records")
    alignment_df.limit(3).show(truncate=False)

print("\n" + "=" * 60)
print("✅ All data is now flattened for better queryability!")
print("=" * 60)

StatementMeta(, 5da9f252-fa2b-4cd6-9eef-c4bbd1ee4c95, 19, Finished, Available, Finished, False)

📊 FLATTENED DATA STRUCTURE EXAMPLES

1️⃣  TENANTS TABLE - Flattened Structure
------------------------------------------------------------
Total columns: 11
Column names (first 20):
  • clientTenantId
  • isBaseline
  • lastBackupTimestamp
  • msTenantId
  • policyDiff
  • recentChanges
  • secureScore
  • tenantDnsName
  • tenantFriendlyName
  • _ingested_at
  • _source_endpoint

Sample data (first 2 records):
-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 clientTenantId      | 14851                                                                                                                                                                                                                                               

In [ ]:
# ============================================
# ASSESSMENTS TABLE - Show Enriched Data
# ============================================

from pyspark.sql import functions as F

print("=" * 60)
print("🔍 ASSESSMENTS TABLE - Enriched with Tenant Details")
print("=" * 60)

if spark.catalog.tableExists("inforcer_assessments"):
    assessments_df = spark.table("inforcer_assessments")
    
    print(f"\nTotal assessments: {assessments_df.count()}")
    print(f"Total columns: {len(assessments_df.columns)}")
    
    # Show key columns including the enriched ones
    key_columns = []
    if "id" in assessments_df.columns:
        key_columns.append("id")
    if "assessed_tenant_name" in assessments_df.columns:
        key_columns.append("assessed_tenant_name")
    if "assessed_tenant_ms_id" in assessments_df.columns:
        key_columns.append("assessed_tenant_ms_id")
    if "assessment_initiated_by" in assessments_df.columns:
        key_columns.append("assessment_initiated_by")
    
    # Add other relevant columns
    for col_name in assessments_df.columns:
        if "tenant" in col_name.lower() and col_name not in key_columns:
            key_columns.append(col_name)
        elif "created" in col_name.lower() or "date" in col_name.lower():
            if col_name not in key_columns:
                key_columns.append(col_name)
    
    # Limit to first 15 columns for display
    display_cols = key_columns[:15] if len(key_columns) > 15 else key_columns
    
    print(f"\nShowing key columns:")
    for column in display_cols:
        print(f"  • {column}")
    
    print("\n" + "-" * 60)
    print("Sample Assessment Records:")
    print("-" * 60)
    
    if display_cols:
        assessments_df.select(display_cols).show(5, truncate=50)
    else:
        assessments_df.show(5, truncate=50)
    
    # Show summary by tenant
    if "assessed_tenant_name" in assessments_df.columns:
        print("\n" + "-" * 60)
        print("Assessments by Tenant:")
        print("-" * 60)
        assessments_df.groupBy("assessed_tenant_name").count().orderBy(F.col("count").desc()).show(10, truncate=False)
    
    # Show summary by initiator
    if "assessment_initiated_by" in assessments_df.columns:
        print("\n" + "-" * 60)
        print("Assessments by Initiator:")
        print("-" * 60)
        assessments_df.groupBy("assessment_initiated_by").count().orderBy(F.col("count").desc()).show(10, truncate=False)
else:
    print("⚠️ Assessments table does not exist")

print("\n" + "=" * 60)

StatementMeta(, 5da9f252-fa2b-4cd6-9eef-c4bbd1ee4c95, 23, Finished, Available, Finished, False)

🔍 ASSESSMENTS TABLE - Enriched with Tenant Details

Total assessments: 6
Total columns: 10

Showing key columns:
  • id
  • assessment_initiated_by
  • created
  • lastUpdated

------------------------------------------------------------
Sample Assessment Records:
------------------------------------------------------------
+--------------------+-----------------------+---------------------------------+---------------------------------+
|                  id|assessment_initiated_by|                          created|                      lastUpdated|
+--------------------+-----------------------+---------------------------------+---------------------------------+
|l1f8wd29pl44pp1j66e3|                   NULL|2025-11-20T17:51:23.7447464+01:00|        2025-11-20T11:00:00+01:00|
|l1f8wd29pl44pp1j632f|                   NULL|2025-07-21T17:51:23.7447464+01:00|2025-10-01T12:58:23.7447463+01:00|
|l1f8wd29pl44pp1j642e|                   NULL|2025-11-19T17:51:23.7447464+01:00|2025-11-19T12:58:23

In [4]:
####################################################################################
# TENANT DIRECTORY REFRESH (self-contained)
# Pulls every customer tenant from Inforcer /beta/tenants (licensed + assessment),
# keeps the DNS name (tenantDnsName), and adds licenseSkus + tenant_type.
# Self-contained so it does not depend on the large definitions cell above.
####################################################################################
import json
import requests
from datetime import datetime, timezone

INF_BASE = "https://api-us.inforcer.com/api"
INF_HEADERS = {"Inf-Api-Key": "0ce9a0ed6d1e4df998e62e57633a4f17"}

# --- 1. Fetch all tenants (endpoint returns the full set in one response) -----------
resp = requests.get(f"{INF_BASE}/beta/tenants", headers=INF_HEADERS, timeout=60)
resp.raise_for_status()
payload = resp.json()
tenants = payload.get("data", []) if isinstance(payload, dict) else (payload or [])
print(f"✅ Retrieved {len(tenants)} tenants from /beta/tenants")


# --- 2. Flatten helper (underscore nested dicts, pipe-delimit primitive arrays) -----
def flatten_record(rec, prefix=""):
    """Return (flat_scalar_dict, {array_name: [objects]}). Follows the ingestion
    preferences: nested dicts -> underscore fields, primitive arrays -> pipe string,
    arrays of objects -> returned separately for extraction into child tables."""
    flat, child_arrays = {}, {}
    for k, v in rec.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            sub_flat, sub_children = flatten_record(v, prefix=f"{key}_")
            flat.update(sub_flat)
            child_arrays.update(sub_children)
        elif isinstance(v, list):
            if len(v) == 0:
                flat[key] = None
            elif all(isinstance(x, dict) for x in v):
                child_arrays[key] = v
            else:
                flat[key] = "|".join(str(x) for x in v)
        else:
            flat[key] = v
    return flat, child_arrays


# --- 3. Build main rows + child tables ----------------------------------------------
main_rows = []
child_tables = {}  # table_name -> list[row]
sync_ts = datetime.now(timezone.utc).isoformat()

for t in tenants:
    licenses = t.get("licenses") or []
    skus = [l.get("sku") for l in licenses if isinstance(l, dict) and l.get("sku")]
    t["licenseSkus"] = "|".join(skus) if skus else None
    t["tenant_type"] = "Licensed" if skus else "Assessment"

    flat, children = flatten_record(t)
    flat["last_sync_timestamp"] = sync_ts
    main_rows.append(flat)

    parent_id = t.get("clientTenantId")
    for arr_name, arr_vals in children.items():
        table_name = f"inforcer_tenant_{arr_name}"
        for item in arr_vals:
            item_flat, _ = flatten_record(item)
            item_flat["parent_id"] = parent_id
            child_tables.setdefault(table_name, []).append(item_flat)

# --- 4. Write main tenants table (JSON read = robust null/type handling) -------------
main_df = spark.read.json(sc.parallelize([json.dumps(r, default=str) for r in main_rows]))
main_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("inforcer_tenants")
print(f"💾 Wrote {main_df.count()} rows to inforcer_tenants ({len(main_df.columns)} columns)")

# --- 5. Write child tables (tags, alignmentSummaries, licenses, ...) -----------------
for table_name, rows in child_tables.items():
    child_df = spark.read.json(sc.parallelize([json.dumps(r, default=str) for r in rows]))
    child_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"💾 Wrote {child_df.count()} rows to {table_name}")

# --- 6. Show the refreshed customer directory ---------------------------------------
print("\n📇 inforcer_tenants — customer directory (licensed + assessment, with DNS):")
spark.table("inforcer_tenants").select(
    "clientTenantId", "tenantFriendlyName", "tenantDnsName", "tenant_type", "licenseSkus"
).orderBy("tenantFriendlyName").show(100, truncate=False)


StatementMeta(, 83a14a2f-cb13-41bf-8324-78adcbce8971, 5, Finished, Available, Finished, False)

✅ Retrieved 15 tenants from /beta/tenants
💾 Wrote 15 rows to inforcer_tenants (14 columns)
💾 Wrote 4 rows to inforcer_tenant_tags
💾 Wrote 14 rows to inforcer_tenant_alignmentSummaries
💾 Wrote 15 rows to inforcer_tenant_licenses

📇 inforcer_tenants — customer directory (licensed + assessment, with DNS):
+--------------+------------------------------------+-------------------------------+-----------+-----------+
|clientTenantId|tenantFriendlyName                  |tenantDnsName                  |tenant_type|licenseSkus|
+--------------+------------------------------------+-------------------------------+-----------+-----------+
|14850         |Admin 365                           |myenterprisegroup.io           |Licensed   |PREMIUM    |
|11057         |Cloudware Limited                   |cloudware.africa               |Licensed   |PREMIUM    |
|11181         |Contoso                             |M365MCP37818446.onmicrosoft.com|Licensed   |PREMIUM    |
|15317         |DANG Lifestyle Inc  

In [5]:
####################################################################################
# CRM <-> INFORCER DOMAIN CORRELATION
# Match Inforcer customer tenants (by DNS name) to Dynamics CRM accounts
# (by primary-contact email domain). Rebuilds dbo.inforcer_crm_correlation.
####################################################################################
from pyspark.sql import functions as F

tenants_df = (
    spark.table("inforcer_tenants")
    .select("clientTenantId", "tenantFriendlyName", "tenantDnsName", "tenant_type", "licenseSkus")
    .withColumn("dns_domain", F.lower(F.trim(F.col("tenantDnsName"))))
)

crm_df = (
    spark.table("crm_accounts")
    .withColumn("crm_domain", F.lower(F.trim(F.element_at(F.split(F.col("PrimaryContactEmail"), "@"), -1))))
    .filter(F.col("crm_domain").isNotNull() & (F.col("crm_domain") != ""))
)

correlation = (
    tenants_df.join(crm_df, tenants_df.dns_domain == crm_df.crm_domain, "left")
    .select(
        tenants_df.clientTenantId,
        tenants_df.tenantFriendlyName,
        tenants_df.tenantDnsName,
        tenants_df.tenant_type,
        tenants_df.licenseSkus,
        crm_df.AccountID,
        crm_df.Name.alias("CrmAccountName"),
        crm_df.Owner,
        crm_df.OwnerEmail,
        crm_df.PrimaryContactEmail,
        F.when(crm_df.AccountID.isNotNull(), F.lit("Matched")).otherwise(F.lit("Unmatched")).alias("match_status"),
    )
)

correlation.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("inforcer_crm_correlation")

total = correlation.count()
matched_tenants = correlation.filter(F.col("match_status") == "Matched").select("clientTenantId").distinct().count()
print(f"💾 inforcer_crm_correlation: {total} rows | {matched_tenants}/15 tenants matched to a CRM account")
print("\n🔗 Matched tenants:")
correlation.filter(F.col("match_status") == "Matched").select(
    "tenantFriendlyName", "tenantDnsName", "CrmAccountName", "Owner", "PrimaryContactEmail"
).orderBy("tenantFriendlyName").show(100, truncate=False)
print("\n⚠️ Unmatched tenants (no CRM account with that email domain):")
correlation.filter(F.col("match_status") == "Unmatched").select(
    "tenantFriendlyName", "tenantDnsName"
).orderBy("tenantFriendlyName").show(100, truncate=False)


StatementMeta(, 83a14a2f-cb13-41bf-8324-78adcbce8971, 6, Finished, Available, Finished, False)

💾 inforcer_crm_correlation: 16 rows | 3/15 tenants matched to a CRM account

🔗 Matched tenants:
+------------------+----------------+----------------------+-------------------+-----------------------------+
|tenantFriendlyName|tenantDnsName   |CrmAccountName        |Owner              |PrimaryContactEmail          |
+------------------+----------------+----------------------+-------------------+-----------------------------+
|Cloudware Limited |cloudware.africa|Cloudware Africa      |Funmilayo Owamokele|Boma@cloudware.africa        |
|Cloudware Limited |cloudware.africa|Cloudware Test Account|Happiness Nwosu    |happiness@cloudware.africa   |
|DANG Lifestyle Inc|danglifestyle.co|Dang Lifestyle Inc.   |Andy Akukwe        |femi@danglifestyle.co        |
|Tenant Admin      |fmdqgroup.com   |FMDQ                  |Sola Jinadu        |Opeyemi.Musibau@fmdqgroup.com|
+------------------+----------------+----------------------+-------------------+-----------------------------+


⚠️ Unmatched t

In [10]:
####################################################################################
# EXTRACT ALL DOMAINS PER TENANT  ->  dbo.inforcer_tenant_domains
# Inforcer only stores ONE primary domain per tenant (tenantDnsName). Additional
# domains are recovered from user UPNs (the part after '@'), which are the tenant's
# verified / onmicrosoft domains. Guests' external home domains are NOT included.
####################################################################################
import json, requests, time
from datetime import datetime, timezone
from pyspark.sql import functions as F

INF_BASE = "https://api-us.inforcer.com/api"
INF_HEADERS = {"Inf-Api-Key": "0ce9a0ed6d1e4df998e62e57633a4f17"}
sync_ts = datetime.now(timezone.utc).isoformat()


def _json_get(path, params=None):
    r = requests.get(f"{INF_BASE}{path}", headers=INF_HEADERS, params=params, timeout=60)
    return r.status_code, (r.json() if r.headers.get("content-type", "").startswith("application/json") else None)


def fetch_users(tenant_id, max_pages=15, page_size=200):
    """Fetch users with defensive pagination + de-dup (endpoint may ignore paging)."""
    out, seen = [], set()
    for page in range(1, max_pages + 1):
        code, body = _json_get(f"/beta/tenants/{tenant_id}/users", {"page": page, "pageSize": page_size})
        users = (body.get("data") if isinstance(body, dict) else body) or []
        if not users:
            break
        new = [u for u in users if u.get("id") not in seen]
        for u in new:
            seen.add(u.get("id"))
        out.extend(new)
        if not new or len(users) < page_size:
            break
        time.sleep(0.7)  # stay within ~90 req/min
    return out


# --- pull tenants ---
_, tbody = _json_get("/beta/tenants")
tenants = tbody.get("data", []) if isinstance(tbody, dict) else tbody
print(f"Processing {len(tenants)} tenants...")

rows = []
for t in tenants:
    tid = t.get("clientTenantId")
    primary = (t.get("tenantDnsName") or "").strip().lower()
    domain_rows = {}  # domain -> source

    if primary:
        domain_rows[primary] = "primary"

    users = fetch_users(tid)
    for u in users:
        upn = (u.get("userPrincipalName") or "")
        if "@" not in upn:
            continue
        dom = upn.split("@")[-1].strip().lower()
        if not dom or dom in domain_rows:
            continue
        domain_rows[dom] = "onmicrosoft" if dom.endswith(".onmicrosoft.com") else "verified_from_users"

    for dom, source in domain_rows.items():
        rows.append({
            "clientTenantId": tid,
            "tenantFriendlyName": t.get("tenantFriendlyName"),
            "tenantPrimaryDns": primary,
            "domain": dom,
            "domain_type": source,
            "is_primary": (dom == primary),
            "user_count_scanned": len(users),
            "last_sync_timestamp": sync_ts,
        })
    print(f"  {t.get('tenantFriendlyName'):<40} {len(domain_rows)} domain(s) from {len(users)} users")

# --- write table ---
domains_df = spark.read.json(sc.parallelize([json.dumps(r, default=str) for r in rows]))
domains_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("inforcer_tenant_domains")
print(f"\n💾 inforcer_tenant_domains: {domains_df.count()} domain rows across {len(tenants)} tenants")

print("\n📇 All domains per tenant:")
domains_df.select("tenantFriendlyName", "domain", "domain_type", "is_primary").orderBy(
    "tenantFriendlyName", F.col("is_primary").desc(), "domain"
).show(200, truncate=False)


StatementMeta(, cb45d43a-8c4d-45df-b0b9-2b2224258114, 7, Finished, Available, Finished, False)

Processing 15 tenants...
  Cloudware Limited                        4 domain(s) from 200 users
  Contoso                                  1 domain(s) from 0 users
  Admin 365                                3 domain(s) from 200 users
  Tenant Admin                             3 domain(s) from 200 users
  Plus Acuity Nigeria                      2 domain(s) from 170 users
  PPL GHANA Limited                        2 domain(s) from 200 users
  LATC Limited                             1 domain(s) from 200 users
  Reliance Software Design                 1 domain(s) from 22 users
  KINETIC TOURS COMPANY LIMITED            2 domain(s) from 22 users
  DANG Lifestyle Inc                       2 domain(s) from 11 users
  Ghana College of Nurses and Midwives     2 domain(s) from 10 users
  RG Estate Management Company             2 domain(s) from 170 users
  Dominion University College              3 domain(s) from 200 users
  University of Gold Coast                 3 domain(s) from 200 users
 

In [ ]:
####################################################################################
# JOIN ALL TENANT DOMAINS -> CRM ACCOUNTS
# A tenant matches if ANY of its domains (primary, onmicrosoft, or verified) hits a
# CRM account's primary-contact email domain. Rebuilds dbo.inforcer_crm_domain_matches
# (row-per-match) and prints a 15-tenant rollup.
####################################################################################
from pyspark.sql import functions as F

domains_df = (
    spark.table("inforcer_tenant_domains")
    .select("clientTenantId", "tenantFriendlyName", "domain", "domain_type", "is_primary")
    .withColumn("domain", F.lower(F.trim(F.col("domain"))))
)

crm_df = (
    spark.table("crm_accounts")
    .withColumn("crm_domain", F.lower(F.trim(F.element_at(F.split(F.col("PrimaryContactEmail"), "@"), -1))))
    .filter(F.col("crm_domain").isNotNull() & (F.col("crm_domain") != ""))
)

# Row-per-match (a domain can match multiple CRM accounts)
matches = domains_df.join(crm_df, domains_df.domain == crm_df.crm_domain, "inner").select(
    domains_df.clientTenantId,
    domains_df.tenantFriendlyName,
    domains_df.domain.alias("matched_domain"),
    domains_df.domain_type,
    domains_df.is_primary,
    crm_df.AccountID,
    crm_df.Name.alias("CrmAccountName"),
    crm_df.Owner,
    crm_df.OwnerEmail,
    crm_df.PrimaryContactEmail,
)
matches.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("inforcer_crm_domain_matches")

# Tenant-level rollup across all 15 tenants
all_tenants = spark.table("inforcer_tenant_domains").select(
    "clientTenantId", "tenantFriendlyName"
).distinct()
matched_roll = (
    matches.groupBy("clientTenantId")
    .agg(
        F.concat_ws(", ", F.collect_set("matched_domain")).alias("matched_domains"),
        F.concat_ws(", ", F.collect_set("CrmAccountName")).alias("crm_accounts"),
        F.countDistinct("AccountID").alias("crm_account_count"),
    )
)
rollup = (
    all_tenants.join(matched_roll, "clientTenantId", "left")
    .withColumn("match_status", F.when(F.col("crm_account_count") > 0, F.lit("Matched")).otherwise(F.lit("Unmatched")))
    .orderBy(F.col("crm_account_count").desc_nulls_last(), "tenantFriendlyName")
)

total_matches = matches.count()
tenants_matched = matched_roll.count()
print(f"💾 inforcer_crm_domain_matches: {total_matches} match rows | {tenants_matched}/15 tenants matched (was 3/15 on primary DNS only)")

print("\n🔗 Tenant-level rollup:")
rollup.select("tenantFriendlyName", "match_status", "matched_domains", "crm_accounts", "crm_account_count").show(50, truncate=False)


StatementMeta(, 57f64927-0a14-478c-aab1-07c44a0b8361, 6, Finished, Available, Finished, False)

💾 inforcer_crm_domain_matches: 4 match rows | 3/15 tenants matched (was 3/15 on primary DNS only)

🔗 Tenant-level rollup:
+------------------------------------+------------+----------------+----------------------------------------+-----------------+
|tenantFriendlyName                  |match_status|matched_domains |crm_accounts                            |crm_account_count|
+------------------------------------+------------+----------------+----------------------------------------+-----------------+
|Cloudware Limited                   |Matched     |cloudware.africa|Cloudware Africa, Cloudware Test Account|2                |
|DANG Lifestyle Inc                  |Matched     |danglifestyle.co|Dang Lifestyle Inc.                     |1                |
|Tenant Admin                        |Matched     |fmdqgroup.com   |FMDQ                                    |1                |
|Admin 365                           |Unmatched   |NULL            |NULL                                    |N

In [2]:
####################################################################################
# DO ASSESSMENT-ONLY CUSTOMERS HAVE A DNS NAME?
# Assessment-only customers come from PDF parsing (security_assessment_*). Check
# (a) whether those tables carry any domain/DNS column, and (b) how many overlap by
# name with the 15 Inforcer tenants (which DO have DNS).
####################################################################################
from pyspark.sql import functions as F

sa = spark.table("security_assessment_assessments")
print("security_assessment_assessments columns:", sa.columns)

# Detect a tenant/customer name column
name_col = next((c for c in sa.columns if any(k in c.lower() for k in ["tenant", "customer", "client", "company", "organization", "org_name"])), None)
domain_cols = [c for c in sa.columns if any(k in c.lower() for k in ["domain", "dns", "url", "website"])]
print(f"Detected name column: {name_col} | domain-like columns: {domain_cols}")

assess_customers = sa.select(F.col(name_col).alias("customer_name")).where(F.col(name_col).isNotNull()).distinct()
n_assess = assess_customers.count()

inf = spark.table("inforcer_tenant_domains").select(
    F.lower(F.trim(F.col("tenantFriendlyName"))).alias("inf_name"),
).distinct()

overlap = (
    assess_customers.withColumn("k", F.lower(F.trim(F.col("customer_name"))))
    .join(inf, F.col("k") == inf.inf_name, "inner")
    .select("customer_name").distinct()
)
n_overlap = overlap.count()

print(f"\nDistinct assessment-only customers: {n_assess}")
print(f"Have a domain-like column in security tables: {'YES -> ' + str(domain_cols) if domain_cols else 'NO'}")
print(f"Name-match to an Inforcer tenant (=> inherit DNS): {n_overlap}")
if n_overlap:
    print("\nThose with DNS via Inforcer name-match:")
    overlap.show(50, truncate=False)


StatementMeta(, 57f64927-0a14-478c-aab1-07c44a0b8361, 8, Finished, Available, Finished, False)

security_assessment_assessments columns: ['assessment_id', 'file_name', 'file_path', 'assessment_type', 'assessment_name', 'tenant_name', 'tenant_assessment_name', 'assessment_date', 'assessment_time', 'overall_score_pct', 'passed_count', 'failed_count', 'warnings_count', 'created_at', 'updated_at', 'ingested_at']
Detected name column: tenant_name | domain-like columns: []

Distinct assessment-only customers: 86
Have a domain-like column in security tables: NO
Name-match to an Inforcer tenant (=> inherit DNS): 6

Those with DNS via Inforcer name-match:
+-------------------+
|customer_name      |
+-------------------+
|Tenant Admin       |
|LATC Limited       |
|Admin 365          |
|DANG Lifestyle Inc |
|Plus Acuity Nigeria|
|PPL GHANA Limited  |
+-------------------+

